In [ ]:
# Create the full dataset without labels

import pandas as pd
from rdkit import Chem
import logging

# --- SETUP LOGGING ---
# Configure logging to capture errors and progress
logging.basicConfig(filename='parse_sdf.log', level=logging.DEBUG,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# --- INITIAL SETUP ---
# File paths
SDF_FILE = r"C:\CKI\dataset\wash-minimized.sdf"  
OUTPUT_FILE = r"C:\CKI\dataset\merged_docking_data.csv"
TARGET_UNIPROT_ID = 'P12532'  # uMtCK

# --- STEP 1: READ SDF FILE ---
print("Reading SDF file...")
logging.debug("Reading SDF file")
try:
    # Initialize RDKit SDF supplier with sanitization disabled to preserve invalid molecules
    supplier = Chem.SDMolSupplier(SDF_FILE, sanitize=False)
    data = []
    for i, mol in enumerate(supplier, 1):  # Start index at 1 for 'number' column
        smiles = None
        zinc_id = None
        wash1_6 = None
        if mol is not None:
            try:
                # Try to get SMILES from property or molecule
                smiles = mol.GetProp('smiles') if mol.HasProp('smiles') else Chem.MolToSmiles(mol, isomericSmiles=True)
                zinc_id = mol.GetProp('zinc_id') if mol.HasProp('zinc_id') else None
                wash1_6 = mol.GetProp('wash1_6') if mol.HasProp('wash1_6') else None
            except Exception as e:
                logging.warning(f"Molecule at index {i} failed processing: {e}")
        else:
            logging.warning(f"Molecule at index {i} is None (invalid coordinates or format)")
        # Append data even if molecule is invalid to preserve order
        data.append({
            'number': i,
            'SMILES': smiles,
            'ZINC_ID': zinc_id,
            'wash1_6': wash1_6,
            'affinity': None  # Empty affinity column
        })
    # Create DataFrame
    df = pd.DataFrame(data)
    # Ensure number is integer
    df['number'] = df['number'].astype('Int64')
    print(f"Loaded SDF with {len(df)} compounds")
    logging.debug(f"Loaded SDF with {len(df)} compounds")
    # Log count of NaN SMILES
    nan_smiles_count = df['SMILES'].isna().sum()
    print(f"Compounds with NaN SMILES: {nan_smiles_count}")
    logging.debug(f"Compounds with NaN SMILES: {nan_smiles_count}")
    if len(df) != 782197:
        print(f"Warning: Expected 782,197 compounds, got {len(df)}")
        logging.warning(f"Expected 782,197 compounds, got {len(df)}")
    if df.empty:
        print("No compounds loaded from SDF. Check file or property names.")
        logging.error("No compounds in SDF")
        exit()
except Exception as e:
    print(f"Error reading SDF file: {e}")
    logging.error(f"Failed to read SDF: {e}")
    exit()

# --- STEP 2: SAVE INITIAL DATASET ---
print("Saving initial dataset...")
try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ Initial dataset saved: {OUTPUT_FILE} ({len(df)} compounds)")
    logging.debug(f"Saved initial dataset to {OUTPUT_FILE}")
except Exception as e:
    print(f"Error saving dataset: {e}")
    logging.error(f"Failed to save dataset: {e}")
    exit()

# --- STEP 3: CHECK DIMENSIONS AND HEAD ---
print("\nDataset Dimensions:", df.shape)
print("\nDataset Head:")
print(df.head())

In [ ]:
# df_all (first 9999 compounds will have affinity values.)
import pandas as pd
import logging

# --- SETUP LOGGING ---
logging.basicConfig(filename='merge_affinity.log', level=logging.DEBUG,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# --- INITIAL SETUP ---
INPUT_FILE = r"C:\CKI\dataset\full-dataset-without-affinity.csv"  
SUMMARY_FILE = r"C:\CKI\dataset\Summary9999.txt"  
OUTPUT_FILE = r"C:\CKI\dataset\df_all.csv"

# --- STEP 1: LOAD FILTERED DATASET ---
print("Loading filtered dataset...")
logging.debug("Loading filtered dataset")
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded dataset with {len(df)} compounds")
    logging.debug(f"Loaded dataset with {len(df)} compounds")
except Exception as e:
    print(f"Error loading dataset: {e}")
    logging.error(f"Failed to load dataset: {e}")
    exit()

# --- STEP 2: READ SUMMARY.TXT ---
print("Reading Summary.txt...")
logging.debug("Reading Summary.txt")
try:
    # Read Summary.txt (space-separated, columns: filename, mode, affinity, rmsd_l, rmsd_u)
    df_summary = pd.read_csv(SUMMARY_FILE, sep='\s+', header=None,
                             names=['filename', 'mode', 'affinity', 'rmsd_l', 'rmsd_u'])
    # Extract number from filename (e.g., 'drug100.pdbqt.txt' -> 100)
    df_summary['number'] = df_summary['filename'].str.extract(r'drug(\d+)\.pdbqt\.txt').astype('Int64')
    # Filter for numbers 1 to 999
    df_summary = df_summary[df_summary['number'].between(1, 9999)][['number', 'affinity']].dropna()
    print(f"Loaded Summary.txt with {len(df_summary)} docking results")
    logging.debug(f"Loaded Summary.txt with {len(df_summary)} docking results")
except Exception as e:
    print(f"Error reading Summary.txt: {e}")
    logging.error(f"Failed to read Summary.txt: {e}")
    exit()

# --- STEP 3: MERGE AFFINITIES ---
print("Merging affinities...")
logging.debug("Merging affinities")
try:
    # Merge on 'number' column
    df_merged = df.merge(df_summary, on='number', how='left')
    # Update affinity column (preserve existing None values for unmatched compounds)
    df['affinity'] = df_merged['affinity_y'].combine_first(df['affinity'])
    print(f"Merged dataset contains {len(df)} compounds")
    logging.debug(f"Merged dataset: {len(df)} compounds")
    # Check for unmatched compounds
    unmatched_count = df['affinity'].isna().sum()
    if unmatched_count > 0:
        print(f"Warning: {unmatched_count} compounds have no affinity data")
        logging.debug(f"Unmatched compounds: {unmatched_count}")
except Exception as e:
    print(f"Error merging affinities: {e}")
    logging.error(f"Failed to merge affinities: {e}")
    exit()

# --- STEP 4: SAVE MERGED DATASET ---
print("Saving merged dataset...")
try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ Merged dataset saved: {OUTPUT_FILE} ({len(df)} compounds)")
    logging.debug(f"Saved merged dataset to {OUTPUT_FILE}")
    # Display dimensions and head
    print("\nDataset Dimensions:", df.shape)
    print("\nDataset Head:")
    print(df.head())
except Exception as e:
    print(f"Error saving merged dataset: {e}")
    logging.error(f"Failed to save merged dataset: {e}")
    exit()

In [ ]:
#final df_all with duplicates and RETAIN_STEREOCHEMISTRY = True

import pandas as pd
from rdkit import Chem
from rdkit.Chem import MolStandardize, SaltRemover
import logging
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- SETUP LOGGING ---
logging.basicConfig(filename='preprocess_verify_duplicates.log', level=logging.DEBUG,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# --- INITIAL SETUP ---
INPUT_FILE = r"C:\CKI\dataset\df_all.csv"  # Input dataset path
OUTPUT_FILE = r"C:\CKI\dataset\df_all_preprocessed_with_duplicates_stereo_true.csv"  # Output dataset path
DUPLICATES_FILE = r"C:\CKI\dataset\df_all_duplicates_stereo.csv"  # Duplicates output path
TARGET_UNIPROT_ID = 'P12532'  # uMtCK
RETAIN_STEREOCHEMISTRY = True  # Set to False to remove stereochemistry

# --- STEP 1: LOAD AND INSPECT DATASET ---
print("Loading dataset...")
logging.debug("Loading dataset")
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded dataset with {len(df)} compounds")
    logging.debug(f"Loaded dataset with {len(df)} compounds")
    # Check initial missing values
    print("\nInitial Missing Values:")
    print(df.isna().sum())
    logging.debug(f"Initial missing values: {df.isna().sum().to_dict()}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    logging.error(f"Failed to load dataset: {e}")
    exit()

# --- STEP 2: HANDLE MISSING VALUES ---
print("\nHandling missing values...")
logging.debug("Handling missing values")
try:
    # Remove rows with NaN SMILES
    initial_len = len(df)
    df = df.dropna(subset=['SMILES'])
    if len(df) < initial_len:
        print(f"Removed {initial_len - len(df)} rows with NaN SMILES")
        logging.debug(f"Removed {initial_len - len(df)} rows with NaN SMILES")
    # Log rows with NaN affinity (keep for now)
    nan_affinity_count = df['affinity'].isna().sum()
    if nan_affinity_count > 0:
        print(f"Found {nan_affinity_count} rows with NaN affinity (retained for now)")
        logging.debug(f"NaN affinity count: {nan_affinity_count}")
except Exception as e:
    print(f"Error handling missing values: {e}")
    logging.error(f"Failed to handle missing values: {e}")
    exit()

# --- STEP 3: VALIDATE SMILES ---
print("\nValidating SMILES...")
logging.debug("Validating SMILES")
try:
    def is_valid_smiles(smiles):
        return Chem.MolFromSmiles(smiles, sanitize=False) is not None

    # Identify invalid SMILES
    invalid_smiles = df[~df['SMILES'].apply(is_valid_smiles)]
    if not invalid_smiles.empty:
        print(f"Found {len(invalid_smiles)} invalid SMILES:")
        print(invalid_smiles[['number', 'SMILES', 'ZINC_ID']])
        logging.debug(f"Invalid SMILES: {invalid_smiles['number'].tolist()}")
        initial_len = len(df)
        df = df[df['SMILES'].apply(is_valid_smiles)]
        print(f"Removed {initial_len - len(df)} rows with invalid SMILES")
        logging.debug(f"Removed {initial_len - len(df)} rows with invalid SMILES")
    else:
        print("No invalid SMILES found")
        logging.debug("No invalid SMILES found")
except Exception as e:
    print(f"Error validating SMILES: {e}")
    logging.error(f"Failed to validate SMILES: {e}")
    exit()

# --- STEP 4: STANDARDIZE SMILES ---
print("\nStandardizing SMILES...")
logging.debug("Standardizing SMILES")
try:
    # Initialize salt remover and normalizer
    remover = SaltRemover.SaltRemover()
    normalizer = MolStandardize.normalize.Normalizer(max_restarts=1000)
    
    def standardize_smiles(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            # Remove salts
            mol = remover.StripMol(mol, dontRemoveEverything=True)
            # Normalize (neutralize charges, standardize tautomers)
            mol = normalizer.normalize(mol)
            # Convert to canonical SMILES
            return Chem.MolToSmiles(mol, isomericSmiles=RETAIN_STEREOCHEMISTRY)
        except Exception as e:
            logging.warning(f"Failed to standardize SMILES: {smiles}, error: {e}")
            return None
    
    initial_len = len(df)
    df['SMILES'] = df['SMILES'].apply(standardize_smiles)
    df = df.dropna(subset=['SMILES'])
    if len(df) < initial_len:
        print(f"Removed {initial_len - len(df)} rows with failed SMILES standardization")
        logging.debug(f"Removed {initial_len - len(df)} rows with failed standardization")
except Exception as e:
    print(f"Error standardizing SMILES: {e}")
    logging.error(f"Failed to standardize SMILES: {e}")
    exit()

# --- STEP 5: IDENTIFY AND SAVE DUPLICATES ---
print("\nIdentifying duplicates...")
logging.debug("Identifying duplicates")
try:
    # Find duplicates based on SMILES
    duplicates = df[df.duplicated(subset='SMILES', keep=False)]
    if not duplicates.empty:
        print(f"Found {len(duplicates)} rows with duplicated SMILES")
        logging.debug(f"Found {len(duplicates)} duplicate rows")
        # Save duplicates for inspection
        duplicates.to_csv(DUPLICATES_FILE, index=False)
        print(f"Saved duplicates to {DUPLICATES_FILE}")
        logging.debug(f"Saved duplicates to {DUPLICATES_FILE}")
        # Print sample of duplicates
        print("\nSample of Duplicates (first 5 groups):")
        duplicate_groups = duplicates.groupby('SMILES').apply(
            lambda x: x[['number', 'SMILES', 'ZINC_ID', 'affinity']].to_dict('records')
        )
        for i, (smiles, group) in enumerate(duplicate_groups.items()):
            if i >= 5: break
            print(f"\nSMILES: {smiles}")
            for row in group:
                print(f"  number: {row['number']}, original SMILES: {row['SMILES']}, ZINC_ID: {row['ZINC_ID']}, affinity: {row['affinity']}")
except Exception as e:
    print(f"Error identifying duplicates: {e}")
    logging.error(f"Failed to identify duplicates: {e}")
    exit()

'''
# --- Let's skip STEP 6: REMOVE DUPLICATES ---
print("\nRemoving duplicates...")
logging.debug("Removing duplicates")
try:
    initial_len = len(df)
    # Group by SMILES, keep first or mean affinity
    df = df.groupby('SMILES').agg({
        'number': 'first',
        'ZINC_ID': 'first',
        'affinity': 'mean'  # Average affinity for duplicates
    }).reset_index()
    if len(df) < initial_len:
        print(f"Removed {initial_len - len(df)} duplicate SMILES")
        logging.debug(f"Removed {initial_len - len(df)} duplicates")
except Exception as e:
    print(f"Error removing duplicates: {e}")
    logging.error(f"Failed to remove duplicates: {e}")
    exit()
'''

# --- STEP 7: FINAL CLEANUP ---
print("\nFinal cleanup...")
logging.debug("Final cleanup")
try:
    # Rename ZINC_ID to Name
    df = df.rename(columns={'ZINC_ID': 'Name'})
    # Ensure number is integer
    df['number'] = df['number'].astype('Int64')
    # Remove wash1_6 if present
    if 'wash1_6' in df.columns:
        df = df.drop(columns=['wash1_6'])
    # Reorder columns
    df = df[['number', 'SMILES', 'Name', 'affinity']]
except Exception as e:
    print(f"Error in final cleanup: {e}")
    logging.error(f"Failed final cleanup: {e}")
    exit()

# --- STEP 8: SAVE PRE-PROCESSED DATASET ---
print("\nSaving pre-processed dataset...")
try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ Pre-processed dataset saved: {OUTPUT_FILE} ({len(df)} compounds)")
    logging.debug(f"Saved pre-processed dataset to {OUTPUT_FILE}")
    # Display final dimensions and head
    print("\nFinal Dataset Dimensions:", df.shape)
    print("\nFinal Dataset Head:")
    print(df.head())
    print("\nFinal Missing Values:")
    print(df.isna().sum())
except Exception as e:
    print(f"Error saving dataset: {e}")
    logging.error(f"Failed to save dataset: {e}")
    exit()